## Transform Circuits Data
1. Read bronze circuits table
2. Keep only the columns required for analytics(Drop url Column)
3. Standardise column names using snake_case (circuitId -> circuit_id, circuitName -> circuit_name)
4. Rename columns to make them more meaningful (lat -> latitude, long->longitude)
5. Filter out rows where circuit_id is null(business key validation)
6. Remove duplicate records
7. Transform values of columns circuit_name and locality to Title Case
8. Write the transformed data to silver circuits table

Below changes are required to implement incremental Load Processing
1. Accept batch_id as a parameter to the notebook
2. Process data for only the batch_id being passes in (i.e.,  filter reading from bronze using the batch_id)
3. Add created_timestamp, updated_timestamp and batch_id to the silver table
4. Merge the processed data to the silver table
    - created_timestamp should be only populated at the time of inserting/creating the record. It should not be updated during the merge update
    - Ensure that we are not overwriting the data in silver table by older bronze data(re-run scenario)

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

#### 1. Read bronze circuits table

In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_df = (
    spark.read.table(bronze_table)
    .filter(F.col("batch_id") == v_batch_id)
    )

#### 2. Keep only the columns required for analytics(Drop url Column)

In [0]:
from pyspark.sql import functions as F
circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

#### Step 3 & 4 - Standardise column names
 - Standardise column names using snake_case (circuitId -> circuit_id, circuitName -> circuit_name)
 - Rename columns to make them more meaningful (lat -> latitude, long->longitude)

In [0]:
circuits_renamed_df = (
    circuits_selected_df
    .withColumnsRenamed({
        "circuitId": "circuit_id",
        "circuitName":"circuit_name",
        "lat":"latitude",
        "long":"longitude"
    })
)

#### Step 5 - Filter out rows where circuit_id is null(business key validation)

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

#### Step 6 - Remove duplicate records

In [0]:
circuits_distinct_df = circuits_valid_df.distinct()

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

#### Step 7 - Transform values of columns circuit_name and locality to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df
    .withColumn("circuit_name",F.initcap(F.col("circuit_name")))
    .withColumn("locality",F.initcap(F.col("locality")))
)

#### Step 8 -  Write the transformed data to silver circuits table

In [0]:
# circuits_final_df = (
#     circuits_final_df
#     .withColumn("created_timestamp",F.current_timestamp())
#     .withColumn("updated_timestamp",F.current_timestamp())
# )

In [0]:
# #tgt = target
# #src = source
# if not spark.catalog.tableExists(silver_table):
#     (
#         circuits_final_df
#             .write
#             .format("delta")
#             .mode("overwrite")
#             .saveAsTable(silver_table)
#     )
# else:
#     from delta.tables import DeltaTable

#     delta_table = DeltaTable.forName(spark, silver_table)
#     (
#             delta_table.alias("tgt").merge(circuits_final_df.alias("src"), "tgt.circuit_id = src.circuit_id")
#             .whenMatchedUpdate(
#                 condition = "src.batch_id >= tgt.batch_id",
#                 set={
#                     "circuit_name": "src.circuit_name",
#                     "latitude": "src.latitude",
#                     "longitude": "src.longitude",
#                     "locality": "src.locality",
#                     "country": "src.country",
#                     "ingestion_timestamp": "src.ingestion_timestamp",
#                     "source_file": "src.source_file",
#                     "batch_id": "src.batch_id",
#                     "updated_timestamp": "src.updated_timestamp"
#                 }
#         )
#         .whenNotMatchedInsertAll()
#         .execute()
#     )

In [0]:
merge_condition = "tgt.circuit_id = src.circuit_id"
columns_to_update=["circuit_id",
                    "latitude",
                    "longitude",
                    "locality",
                    "country",
                    "ingestion_timestamp",
                    "source_file",
                    "batch_id"]

write_to_silver(circuits_final_df, silver_table, merge_condition, columns_to_update)